In [5]:
#!pip install -q langchain langchain-community sentence-transformers chromadb groq langchain-groq

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq

In [7]:
doc = """Artificial intelligence is transforming healthcare.
Doctors now use AI to detect cancer early from scans.
Meanwhile, AI is also changing finance.
Banks use machine learning to detect fraud in real time.
Education is another field being disrupted.
Personalized AI tutors can adapt to each student's learning pace."""

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20) # 100 characters.

In [9]:
docs = splitter.split_text(doc)

In [10]:
docs

['Artificial intelligence is transforming healthcare.',
 'Doctors now use AI to detect cancer early from scans.\nMeanwhile, AI is also changing finance.',
 'Banks use machine learning to detect fraud in real time.',
 'Education is another field being disrupted.',
 "Personalized AI tutors can adapt to each student's learning pace."]

In [11]:
model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

/tmp/ipykernel_11634/3618964508.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
vectorstore = Chroma.from_texts(texts=docs, embedding=model)

In [13]:
vectorstore.similarity_search('How is AI used in space?')

[Document(metadata={}, page_content='Doctors now use AI to detect cancer early from scans.\nMeanwhile, AI is also changing finance.'),
 Document(metadata={}, page_content="Personalized AI tutors can adapt to each student's learning pace."),
 Document(metadata={}, page_content='Artificial intelligence is transforming healthcare.'),
 Document(metadata={}, page_content='Banks use machine learning to detect fraud in real time.')]

In [14]:
llm = ChatGroq(api_key="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", model_name="llama-3.1-8b-instant")

In [15]:
retriever = vectorstore.as_retriever()

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [17]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context: {context}

Question: {question}
""")

- retriever takes that query, searches ChromaDB, returns relevant chunks → goes into {context}
- RunnablePassthrough takes that same query and passes it as is → goes into {question}

In [18]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)

In [19]:
response = chain.invoke("How does AI help doctors?")
print(response.content)

Doctors now use AI to detect cancer early from scans.


**When you call chain.invoke("How does AI help doctors?") here's what happens step by step:**

- Your question enters the chain
- retriever searches ChromaDB and finds the most relevant chunk → fills {context}
- RunnablePassthrough passes your question as is → fills {question}
- prompt combines both into a full prompt message
- llm (your ChatGroq) receives that prompt and generates an answer
- Answer comes back to you

In [20]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 6.2 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://en.wikipedia.org/wiki/Project_Hail_Mary_(film)")
pages = loader.load()


In [30]:
web_splits = splitter.split_documents(pages) # SPLITTING
vectorstore_web = Chroma.from_documents(documents=web_splits, embedding=model) #Storing
retriver_web = vectorstore_web.as_retriever()


In [33]:
chain=(
    {"context":retriver_web, "question":RunnablePassthrough()}
    | prompt
    | llm
)

In [34]:
response = chain.invoke('Who is the director of Project Hail Mary?')
print(response.content)

Phil Lord


In [42]:
response = chain.invoke("Who invented the telephone?")
print(response.content)

I don't know.


In [43]:
docs = retriver_web.invoke("What is Project Hail Mary about?")
for doc in docs:
    print(doc.metadata)
    print(doc.page_content[:100])
    print("---")

{'source': 'https://en.wikipedia.org/wiki/Project_Hail_Mary_(film)', 'title': 'Project Hail Mary (film) - Wikipedia', 'language': 'en'}
Project Hail Mary (film) - Wikipedia






























Jump to content
---
{'language': 'en', 'source': 'https://en.wikipedia.org/wiki/Project_Hail_Mary_(film)', 'title': 'Project Hail Mary (film) - Wikipedia'}
from "https://en.wikipedia.org/w/index.php?title=Project_Hail_Mary_(film)&oldid=1351258387"
---
{'title': 'Project Hail Mary (film) - Wikipedia', 'source': 'https://en.wikipedia.org/wiki/Project_Hail_Mary_(film)', 'language': 'en'}
what also makes Project Hail Mary so gosh darn lovable is that it imbues us with child-like wonder
---
{'language': 'en', 'source': 'https://en.wikipedia.org/wiki/Project_Hail_Mary_(film)', 'title': 'Project Hail Mary (film) - Wikipedia'}
External links[edit]



Wikimedia Commons has media related to Project Hail Mary (film).
---
